# D-04 Data Preprocessing — Migrants by Educational Level (Census 2011)

**File:** DS-0000-D04-MDDS.XLSX  
**What this dataset has:** For each state, migrants broken down by place of last residence, age group, sex, **educational level**, and duration of residence.

**What we need:** A single summary row per state (and one for India) showing total migrant counts by literacy and education level — no age group, no duration, no last residence, no rural/urban split.

---

### Raw Structure (30,791 rows × 32 columns)

Each state has **855 rows** due to multiple combinations of:
- Place of Enumeration: Total / Rural / Urban (col E)
- Duration of Residence: 5 categories (col F)
- Age Group: 19 categories (col G)
- Last Residence: Total / Rural / Urban (col H)

We only want the single row per state where **all four** filters are at their top-level 'Total / All' value.

---

### Preprocessing Plan

**Columns to KEEP (Excel D, L → AF):**
- D  → `AreaName` (destination state)
- L–N → Illiterate (Persons, Males, Females)
- O–Q → Literate (Persons, Males, Females)
- R–T → Literate but below Matric/Secondary
- U–W → Matric/Secondary but below Graduate
- X–Z → Technical Diploma (not equal to degree)
- AA–AC → Graduate and above (other than technical degree)
- AD–AF → Technical Degree equal to degree or post-graduate

**Columns to DROP:**
- A → Table Name
- B → State Code
- C → District Code
- E → Place of Enumeration (Total/Rural/Urban)
- F → Duration of Residence
- G → Age Group
- H → Last Residence (Total/Rural/Urban)
- I–K → Total Migrants (Persons, Males, Females)

**Rows to KEEP:**
Only the **first occurrence** per state — where col E = Total, col F = 'All durations of residence', col G = 'All ages', col H = Total. This gives exactly **36 rows** (35 states/UTs + India).

---
## Step 0 — Install & Import Libraries

In [ ]:
!pip install openpyxl --quiet

import pandas as pd
import numpy as np
import re

print("Libraries loaded successfully.")

---
## Step 1 — Upload & Load Raw Dataset

In [ ]:
from google.colab import files

print("Please upload DS-0000-D04-MDDS.XLSX")
uploaded = files.upload()

In [ ]:
# Load with no header — read everything as-is
df_raw = pd.read_excel('DS-0000-D04-MDDS.XLSX', header=None)

print(f"Raw shape  : {df_raw.shape}  ({df_raw.shape[0]} rows × {df_raw.shape[1]} columns)")
print()
print("First 7 rows (header block + first 2 data rows):")
df_raw.head(7)

---
## Step 2 — Inspect Column Structure

The first 5 rows are multi-row merged headers. We print them here to confirm every column position.

In [ ]:
print("=== RAW HEADER BLOCK (rows 1–4) ===")
print(df_raw.iloc[1:5, :].to_string())
print()

print("=== COLUMN POSITION MAP ===")
col_map = {
    0:  'A  - Table Name                          → DROP',
    1:  'B  - State Code                          → DROP',
    2:  'C  - District Code                       → DROP',
    3:  'D  - Area Name                           → KEEP',
    4:  'E  - Place of Enumeration (TRU)          → FILTER then DROP',
    5:  'F  - Duration of Residence               → FILTER then DROP',
    6:  'G  - Age Group                           → FILTER then DROP',
    7:  'H  - Last Residence (TRU)                → FILTER then DROP',
    8:  'I  - Total Migrants Persons              → DROP',
    9:  'J  - Total Migrants Males                → DROP',
    10: 'K  - Total Migrants Females              → DROP',
    11: 'L  - Illiterate Persons                  → KEEP',
    12: 'M  - Illiterate Males                    → KEEP',
    13: 'N  - Illiterate Females                  → KEEP',
    14: 'O  - Literate Persons                    → KEEP',
    15: 'P  - Literate Males                      → KEEP',
    16: 'Q  - Literate Females                    → KEEP',
    17: 'R  - Below Matric Persons                → KEEP',
    18: 'S  - Below Matric Males                  → KEEP',
    19: 'T  - Below Matric Females                → KEEP',
    20: 'U  - Matric/Sec but below Grad Persons   → KEEP',
    21: 'V  - Matric/Sec but below Grad Males     → KEEP',
    22: 'W  - Matric/Sec but below Grad Females   → KEEP',
    23: 'X  - Tech Diploma (not degree) Persons   → KEEP',
    24: 'Y  - Tech Diploma (not degree) Males     → KEEP',
    25: 'Z  - Tech Diploma (not degree) Females   → KEEP',
    26: 'AA - Graduate and above Persons          → KEEP',
    27: 'AB - Graduate and above Males            → KEEP',
    28: 'AC - Graduate and above Females          → KEEP',
    29: 'AD - Tech Degree (equal/PG) Persons      → KEEP',
    30: 'AE - Tech Degree (equal/PG) Males        → KEEP',
    31: 'AF - Tech Degree (equal/PG) Females      → KEEP',
}
for idx, desc in col_map.items():
    print(f"  Col {idx:2d} ({chr(65+idx) if idx < 26 else 'A'+chr(65+idx-26)}) : {desc}")

---
## Step 3 — Extract Data Rows & Assign Working Column Names

Skip rows 0–5 (title row + 4 header rows + 1 blank/index row). Assign readable names to all 32 columns.

In [ ]:
# Skip first 6 rows (0-indexed: row 0 = title, rows 1-4 = headers, row 5 = col number index)
df = df_raw.iloc[6:].reset_index(drop=True)
print(f"Data rows after skipping header block: {len(df)}")

# Assign column names matching the Excel column structure
col_names = [
    # Cols A–K : identifiers + filters + total (all will be dropped)
    'TableName',            # col 0  A
    'StateCode',            # col 1  B
    'DistrictCode',         # col 2  C
    'AreaName',             # col 3  D  ← KEEP
    'PlaceOfEnum_TRU',      # col 4  E  ← filter to Total, then drop
    'Duration',             # col 5  F  ← filter to All durations, then drop
    'AgeGroup',             # col 6  G  ← filter to All ages, then drop
    'LastRes_TRU',          # col 7  H  ← filter to Total, then drop
    'TotalMig_Persons',     # col 8  I  ← DROP
    'TotalMig_Males',       # col 9  J  ← DROP
    'TotalMig_Females',     # col 10 K  ← DROP

    # Cols L–AF : Education level breakdown (all KEEP)
    'Illiterate_Persons',   # col 11 L
    'Illiterate_Males',     # col 12 M
    'Illiterate_Females',   # col 13 N

    'Literate_Persons',     # col 14 O
    'Literate_Males',       # col 15 P
    'Literate_Females',     # col 16 Q

    'BelowMatric_Persons',  # col 17 R  (Literate but below Matric/Secondary)
    'BelowMatric_Males',    # col 18 S
    'BelowMatric_Females',  # col 19 T

    'MatricToGrad_Persons', # col 20 U  (Matric/Secondary but below Graduate)
    'MatricToGrad_Males',   # col 21 V
    'MatricToGrad_Females', # col 22 W

    'TechDiploma_Persons',  # col 23 X  (Technical diploma, not equal to degree)
    'TechDiploma_Males',    # col 24 Y
    'TechDiploma_Females',  # col 25 Z

    'Graduate_Persons',     # col 26 AA (Graduate and above, other than technical degree)
    'Graduate_Males',       # col 27 AB
    'Graduate_Females',     # col 28 AC

    'TechDegree_Persons',   # col 29 AD (Technical degree equal to degree or post-graduate)
    'TechDegree_Males',     # col 30 AE
    'TechDegree_Females',   # col 31 AF
]

df.columns = col_names
print(f"Column names assigned : {len(col_names)}")
print()
print("First 3 data rows:")
df.head(3)

---
## Step 4 — Filter to One Row Per State

Each state has 855 rows due to all combinations of TRU, duration, age group and last residence.
We isolate only the single top-level summary row per state:

| Column | Keep value |
|--------|------------|
| E — PlaceOfEnum_TRU | `Total` |
| F — Duration | `All durations of residence` |
| G — AgeGroup | `All ages` |
| H — LastRes_TRU | `Total` |

This will give exactly **36 rows** — one for India + one per 35 states/UTs.

In [ ]:
rows_before = len(df)

# Strip whitespace from all filter columns before comparing
for col in ['PlaceOfEnum_TRU', 'Duration', 'AgeGroup', 'LastRes_TRU']:
    df[col] = df[col].astype(str).str.strip()

# Apply all four filters simultaneously
mask = (
    (df['PlaceOfEnum_TRU'] == 'Total') &
    (df['Duration']        == 'All durations of residence') &
    (df['AgeGroup']        == 'All ages') &
    (df['LastRes_TRU']     == 'Total')
)

df = df[mask].reset_index(drop=True)

print(f"Rows before filtering : {rows_before:,}")
print(f"Rows after filtering  : {len(df)}  ← one per state/UT + India")
print()
print("AreaName values in filtered dataset:")
for name in df['AreaName'].tolist():
    print(f"  {name}")

---
## Step 5 — Drop Unwanted Columns (A, B, C, E, F, G, H, I, J, K)

In [ ]:
cols_to_drop = [
    'TableName', 'StateCode', 'DistrictCode',   # A, B, C
    'PlaceOfEnum_TRU', 'Duration',              # E, F  (already filtered)
    'AgeGroup', 'LastRes_TRU',                  # G, H  (already filtered)
    'TotalMig_Persons', 'TotalMig_Males', 'TotalMig_Females',  # I, J, K
]

df = df.drop(columns=cols_to_drop)

print(f"Columns remaining : {len(df.columns)}")
print()
print("Final column list:")
for i, col in enumerate(df.columns):
    print(f"  {i+1:2d}. {col}")

---
## Step 6 — Clean `AreaName` Column

Strip the `'State - ... (01)'` format, title-case, and fix known abbreviations — matching the naming convention used in D01/D02/D12.

In [ ]:
def clean_area_name(name):
    """Remove 'State - ' prefix and ' (XX)' state code suffix. Title-case the result."""
    name = str(name).strip()
    # Remove 'State - ' prefix
    name = re.sub(r'^State\s*-\s*', '', name, flags=re.IGNORECASE)
    # Remove trailing state code like ' (01)'
    name = re.sub(r'\s*\(\d+\)\s*$', '', name)
    # Title case
    name = name.strip().title()
    # Fix known abbreviations that title-case breaks
    name = name.replace('Nct Of Delhi', 'NCT of Delhi')
    name = name.replace('A & N Islands', 'Andaman & Nicobar Islands')
    name = name.replace('D & N Haveli', 'Dadra & Nagar Haveli')
    return name

df['AreaName'] = df['AreaName'].apply(clean_area_name)

print("Cleaned AreaName values:")
for name in df['AreaName'].tolist():
    print(f"  {name}")

---
## Step 7 — Convert Numeric Columns to Integer

In [ ]:
numeric_cols = [c for c in df.columns if c != 'AreaName']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

print(f"Numeric columns converted : {len(numeric_cols)}")
print(f"Remaining null values     : {df.isnull().sum().sum()}")
print()
print(df.dtypes)

---
## Step 8 — Final Validation

In [ ]:
print("=" * 55)
print("  FINAL CLEANED DATASET — D04")
print("=" * 55)
print(f"Shape          : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Null values    : {df.isnull().sum().sum()}")
print(f"Unique states  : {df['AreaName'].nunique()}  (includes India)")
print()

# Sanity check: Literate + Illiterate should roughly cover all migrants
india_row = df[df['AreaName'] == 'India'].iloc[0]
total_check = india_row['Illiterate_Persons'] + india_row['Literate_Persons']
print(f"India — Illiterate Persons : {india_row['Illiterate_Persons']:,}")
print(f"India — Literate Persons   : {india_row['Literate_Persons']:,}")
print(f"India — Combined Total     : {total_check:,}")
print()
print("Full dataset:")
df

---
## Step 9 — Export & Download

In [ ]:
output_filename = 'D04_cleaned.csv'
df.to_csv(output_filename, index=False)

print(f"Saved → {output_filename}")
print(f"Shape  : {df.shape}")
print(f"Size   : {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

In [ ]:
from google.colab import files
files.download('D04_cleaned.csv')
print("Download triggered for D04_cleaned.csv")